In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

appointments = pd.read_csv(PROCESSED_DIR / "cleaned_appointments.csv")

print("Shape:", appointments.shape)
appointments.head()

Shape: (6837, 29)


,appointment_id,slot_id,scheduling_date,appointment_date,appointment_time,scheduling_interval,status,check_in_time,appointment_duration,start_time,...,appointment_month,appointment_month_name,appointment_day_name,appointment_hour,is_attended,is_no_show,is_cancelled,waiting_time_category,doctor_name,max_appointments_per_day
0,6,1,2023-12-08,2024-01-01,08:00:00,24,attended,07:35:33,46.5,08:01:05,...,1,January,Monday,8,1,0,0,16-30 min,Doctor 10,24
1,118,21,2023-12-25,2024-01-01,13:00:00,7,did not attend,NaN,NaN,NaN,...,1,January,Monday,13,0,1,0,Not attended,Doctor 24,20
2,194,22,2023-12-29,2024-01-01,13:15:00,3,attended,13:01:40,16.9,13:28:46,...,1,January,Monday,13,1,0,0,16-30 min,Doctor 18,20
3,212,23,2023-12-30,2024-01-01,13:30:00,2,attended,13:09:37,8.3,13:46:43,...,1,January,Monday,13,1,0,0,31-45 min,Doctor 16,28
4,168,24,2023-12-28,2024-01-01,13:45:00,4,attended,13:16:48,21.8,13:55:56,...,1,January,Monday,13,1,0,0,31-45 min,Doctor 5,16


In [2]:
appointments.info()
appointments["status"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 6837 entries, 0 to 6836
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appointment_id            6837 non-null   int64  
 1   slot_id                   6837 non-null   int64  
 2   scheduling_date           6837 non-null   str    
 3   appointment_date          6837 non-null   str    
 4   appointment_time          6837 non-null   str    
 5   scheduling_interval       6837 non-null   int64  
 6   status                    6837 non-null   str    
 7   check_in_time             5182 non-null   str    
 8   appointment_duration      5182 non-null   float64
 9   start_time                5182 non-null   str    
 10  end_time                  5182 non-null   str    
 11  waiting_time              5182 non-null   float64
 12  patient_id                6837 non-null   int64  
 13  sex                       6837 non-null   str    
 14  age                

status
attended          5182
cancelled         1096
did not attend     407
scheduled          131
unknown             21
Name: count, dtype: int64

In [3]:
# Use only attended appointments for waiting time analysis
attended = appointments[appointments["status"] == "attended"].copy()

print("Attended appointments:", attended.shape)
print("Average waiting time:", round(attended["waiting_time"].mean(), 2))
print("Median waiting time:", round(attended["waiting_time"].median(), 2))
print("Max waiting time:", round(attended["waiting_time"].max(), 2))

Attended appointments: (5182, 29)
Average waiting time: 26.25
Median waiting time: 17.85
Max waiting time: 174.5


In [4]:
waiting_by_department = (
    attended.groupby("department")
    .agg(
        total_attended=("appointment_id", "count"),
        avg_waiting_time=("waiting_time", "mean"),
        median_waiting_time=("waiting_time", "median"),
        max_waiting_time=("waiting_time", "max"),
        avg_duration=("appointment_duration", "mean")
    )
    .reset_index()
    .sort_values("avg_waiting_time", ascending=False)
)

waiting_by_department

,department,total_attended,avg_waiting_time,median_waiting_time,max_waiting_time,avg_duration
1,Dermatology,625,26.864000,18.1,138.8,17.588320
0,Cardiology,783,26.797190,18.4,166.2,16.979438
5,Pediatrics,871,26.693226,18.6,151.9,17.258553
2,General Medicine,1542,26.244163,18.1,171.7,16.830804
4,Orthopedics,835,25.948144,17.2,174.5,16.668383
3,Neurology,526,24.419202,16.4,126.5,18.100760


In [5]:
doctor_analysis = (
    appointments.groupby(["doctor_id", "doctor_name", "department", "max_appointments_per_day"])
    .agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_attended", "sum"),
        no_shows=("is_no_show", "sum"),
        cancellations=("is_cancelled", "sum"),
        avg_waiting_time=("waiting_time", "mean"),
        avg_duration=("appointment_duration", "mean")
    )
    .reset_index()
)

doctor_analysis["no_show_rate"] = doctor_analysis["no_shows"] / doctor_analysis["total_appointments"]
doctor_analysis["attendance_rate"] = doctor_analysis["attended_appointments"] / doctor_analysis["total_appointments"]

doctor_analysis = doctor_analysis.sort_values("avg_waiting_time", ascending=False)

doctor_analysis.head(10)

,doctor_id,doctor_name,department,max_appointments_per_day,total_appointments,attended_appointments,no_shows,cancellations,avg_waiting_time,avg_duration,no_show_rate,attendance_rate
4,D005,Doctor 5,Dermatology,16,304,243,17,39,30.132922,17.637037,0.055921,0.799342
10,D011,Doctor 11,General Medicine,24,343,256,22,58,28.758594,16.331641,0.064140,0.746356
2,D003,Doctor 3,Cardiology,16,268,193,23,48,28.154922,16.026943,0.085821,0.720149
20,D021,Doctor 21,Pediatrics,16,305,228,24,47,27.764035,17.255263,0.078689,0.747541
1,D002,Doctor 2,Cardiology,16,249,180,17,44,27.682778,17.743889,0.068273,0.722892
19,D020,Doctor 20,Pediatrics,20,290,220,19,44,27.531818,17.615455,0.065517,0.758621
0,D001,Doctor 1,Cardiology,24,265,206,15,40,27.458738,17.836408,0.056604,0.777358
14,D015,Doctor 15,Orthopedics,16,295,225,16,50,27.380889,16.063556,0.054237,0.762712
7,D008,Doctor 8,General Medicine,16,354,256,21,64,27.162109,17.808203,0.059322,0.723164
9,D010,Doctor 10,General Medicine,24,288,216,13,53,26.687963,16.691667,0.045139,0.750000


In [6]:
waiting_by_hour = (
    attended.groupby("appointment_hour")
    .agg(
        total_attended=("appointment_id", "count"),
        avg_waiting_time=("waiting_time", "mean"),
        median_waiting_time=("waiting_time", "median")
    )
    .reset_index()
    .sort_values("appointment_hour")
)

waiting_by_hour

,appointment_hour,total_attended,avg_waiting_time,median_waiting_time
0,8,576,13.248785,9.85
1,9,580,19.541207,12.90
2,10,578,24.210208,17.25
3,11,571,26.970228,19.90
4,12,590,29.401356,23.60
5,13,569,30.024780,22.30
6,14,570,28.945088,19.30
7,15,568,30.592254,20.85
8,16,580,33.339655,24.45


In [7]:
no_show_by_department = (
    appointments.groupby("department")
    .agg(
        total_appointments=("appointment_id", "count"),
        no_shows=("is_no_show", "sum"),
        attended=("is_attended", "sum"),
        cancelled=("is_cancelled", "sum")
    )
    .reset_index()
)

no_show_by_department["no_show_rate"] = (
    no_show_by_department["no_shows"] / no_show_by_department["total_appointments"]
)

no_show_by_department = no_show_by_department.sort_values("no_show_rate", ascending=False)

no_show_by_department

,department,total_appointments,no_shows,attended,cancelled,no_show_rate
0,Cardiology,1056,77,783,174,0.072917
1,Dermatology,820,52,625,125,0.063415
4,Orthopedics,1105,67,835,187,0.060633
2,General Medicine,2034,121,1542,320,0.059489
5,Pediatrics,1152,64,871,191,0.055556
3,Neurology,670,26,526,99,0.038806


In [8]:
waiting_by_department.to_csv(PROCESSED_DIR / "waiting_by_department.csv", index=False)
doctor_analysis.to_csv(PROCESSED_DIR / "doctor_workload_analysis.csv", index=False)
waiting_by_hour.to_csv(PROCESSED_DIR / "waiting_by_hour.csv", index=False)
no_show_by_department.to_csv(PROCESSED_DIR / "no_show_by_department.csv", index=False)

# Main file for dashboard
appointments.to_csv(PROCESSED_DIR / "waiting_time_analysis.csv", index=False)

print("Analysis files saved successfully.")

Analysis files saved successfully.
